Carga de datos

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from lightgbm import LGBMClassifier

train = pd.read_json("data/train_data.jsonlines", lines=True)
print(train.shape)

(70000, 45)


Armado de features

In [3]:
def armar_features(df):
    X = pd.DataFrame(index=df.index)
    for c in ["price", "base_price", "initial_quantity", "sold_quantity", "available_quantity"]:
        X[c] = df[c]
    X["accepts_mercadopago"] = df["accepts_mercadopago"].astype(int)
    X["automatic_relist"]    = df["automatic_relist"].astype(int)
    X["free_shipping"] = df["shipping"].apply(lambda s: int(bool(s.get("free_shipping"))))
    X["local_pick_up"] = df["shipping"].apply(lambda s: int(bool(s.get("local_pick_up"))))
    X["n_pictures"]    = df["pictures"].apply(len)
    X["n_variations"]  = df["variations"].apply(len)
    X["n_attributes"]  = df["attributes"].apply(len)
    X["has_warranty"]  = df["warranty"].notna().astype(int)
    X["listing_type_id"] = df["listing_type_id"]
    X["buying_mode"]     = df["buying_mode"]
    X["shipping_mode"]   = df["shipping"].apply(lambda s: s.get("mode"))
    return X

X = armar_features(train)
y = (train["condition"] == "used").astype(int)   # 1 = usado, 0 = nuevo

CAT = ["listing_type_id", "buying_mode", "shipping_mode"]
NUM = [c for c in X.columns if c not in CAT]
X.head()

,price,base_price,initial_quantity,sold_quantity,available_quantity,accepts_mercadopago,automatic_relist,free_shipping,local_pick_up,n_pictures,n_variations,n_attributes,has_warranty,listing_type_id,buying_mode,shipping_mode
0,15.02,15.02,1,0,1,1,0,0,1,5,0,0,1,bronze,buy_it_now,not_specified
1,5520.00,5520.00,4,0,4,1,0,0,1,1,0,0,1,bronze,buy_it_now,not_specified
2,100.00,100.00,1,0,1,1,0,0,1,6,1,2,0,free,buy_it_now,not_specified
3,65.00,65.00,1,0,1,1,0,0,0,1,1,2,1,free,buy_it_now,custom
4,45.00,45.00,9,1,8,1,0,0,1,0,0,0,0,bronze,buy_it_now,not_specified


Evaluación modelos

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluar(nombre, modelo, XX):
    s = cross_val_score(modelo, XX, y, cv=cv, scoring="accuracy")
    print(f"{nombre:40s} accuracy = {s.mean():.4f} ± {s.std():.4f}")

# 1. Clase mayoritaria
evaluar("Clase mayoritaria", DummyClassifier(), X[NUM])

# 2. Regresión logística: necesita escalar y pasar categorías a columnas 0/1
X_log = X.copy()
for c in ["price", "base_price", "initial_quantity", "sold_quantity", "available_quantity"]:
    X_log[c] = np.log1p(X_log[c])
prepro = make_column_transformer(
    (OneHotEncoder(handle_unknown="ignore"), CAT),
    (StandardScaler(), NUM),
)
evaluar("Regresión logística", make_pipeline(prepro, LogisticRegression(max_iter=2000)), X_log)

# 3. LightGBM: maneja categorías por su cuenta
X_lgb = X.copy()
for c in CAT:
    X_lgb[c] = X_lgb[c].astype("category")
evaluar("LightGBM por defecto", LGBMClassifier(random_state=42, verbose=-1), X_lgb)

Clase mayoritaria                        accuracy = 0.5370 ± 0.0000
Regresión logística                      accuracy = 0.8229 ± 0.0025
LightGBM por defecto                     accuracy = 0.8417 ± 0.0023


Primera submission

In [ ]:
import os

test = pd.read_json("data/test_data.jsonlines", lines=True)
X_test = armar_features(test)

# Las categorías de test tienen que ser las mismas que en train
for c in CAT:
    X_test[c] = pd.Categorical(X_test[c], categories=X_lgb[c].cat.categories)

# Entrenar con TODO train (antes era solo para evaluar)
modelo = LGBMClassifier(random_state=42, verbose=-1)
modelo.fit(X_lgb, y)

pred = modelo.predict(X_test)
sub = pd.DataFrame({
    "ID": range(len(test)),
    "condition": np.where(pred == 1, "used", "new"),
})

os.makedirs("submissions", exist_ok=True)
sub.to_csv("submissions/01_baseline_lgbm.csv", index=False)

print(sub.shape)
sub.head()

(30000, 2)


,ID,condition
0,MLA8021791613,used
1,MLA8303606049,used
2,MLA7121650694,used
3,MLA2644578345,used
4,MLA9767546656,new
